<h1> Well coordinates and production dispensing <h1>

Texas do not have well level data. But we have found how many well are there per well. We can make an approximation that all wells in a lease produce the same volume by dividing the total production per lease by number of wells and associating it with each lease. This notebook assign coordinates with production_dispensing data and make it a proxy well level data by performing what was stated above

In [1]:
from pathlib import Path
import re
import zipfile
import warnings

import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

In [2]:
production_disp_path = Path("../../../../data/raw/texas/cleaned_data/texas_prod_disp.parquet")
well_coord_path = Path("../../../../data/raw/texas/cleaned_data/lease_well_coordinates.geoparquet")
shapefile_zip_folder = Path("../../../../data/raw/texas/Wells")

output_folder = Path("../../../../data/raw/texas/cleaned_data")
output_folder.mkdir(parents=True, exist_ok=True)

In [3]:
prod_disp = pd.read_parquet(production_disp_path)
well_coord = gpd.read_parquet(well_coord_path)

print("Production shape:", prod_disp.shape)
print("Lease/well shape:", well_coord.shape)

print("\nProduction columns:")
print(prod_disp.columns.tolist())

print("\nLease/well columns:")
print(well_coord.columns.tolist())

Production shape: (16761949, 29)
Lease/well shape: (587614, 32)

Production columns:
['oil_gas_code', 'district_no', 'lease_no', 'field_no', 'operator_no', 'operator_name', 'oil_pipeline_bbl', 'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl', 'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl', 'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl', 'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf', 'csgd_transmission_mcf', 'csgd_processing_plant_mcf', 'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf', 'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf', 'csgd_no_disp_code_mcf', 'oil_sold_total_bbl', 'total_vented_flared_mcf', 'date']

Lease/well columns:
['oil_gas_code', 'district_no', 'lease_no', 'well_no', 'api_county_code', 'api_unique_no', 'county_name', 'wellbore_location_code', 'api_no', 'oil_gas_code_norm', 'district_no_norm', 'lease_no_norm', 'lease_key', 'api8_from_components', 'api8_from_api_no', 'api8', 'longitude', 'lat

In [4]:
prod_disp.date.tail(5)

16761944   1993-02-01
16761945   1993-03-01
16761946   1993-04-01
16761947   1993-05-01
16761948   1993-06-01
Name: date, dtype: datetime64[us]

In [5]:
prod_disp = prod_disp.loc[prod_disp['date']> '2011-01-01']
prod_disp.shape

(7994195, 29)

In [6]:
prod_disp.columns = prod_disp.columns.str.lower()
well_coord.lease_key.head(5)

0    O_08_00277
1    O_08_00277
2    O_08_00287
3    O_08_00291
4    O_08_00291
Name: lease_key, dtype: string

Production file is lease-level and lease/well file links leases to API numbers. We need a stable lease key to identify them between files. We do oil/gas code + district code + lease code.

In [7]:
def clean_text_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )

def normalize_district(s):
    s = clean_text_series(s)
    
    # RRC districts can be 01, 02, 03, 04, 05, 06, 7B, 7C, 08, 8A, 09, 10, etc.
    # If purely numeric, pad to 2 digits. Leave 8A, 7B, etc. as-is.
    return s.apply(lambda x: x.zfill(2) if pd.notna(x) and x.isdigit() else x)

def normalize_lease_no(s):
    s = clean_text_series(s)
    
    # Many RRC lease numbers are 5 digits.
    # If your source uses a different convention, inspect before changing.
    return s.apply(lambda x: x.zfill(5) if pd.notna(x) and x.isdigit() else x)

def normalize_oil_gas_code(s):
    return clean_text_series(s)

In [8]:
for df in [prod_disp]:
    df["oil_gas_code_norm"] = normalize_oil_gas_code(df["oil_gas_code"])
    df["district_no_norm"] = normalize_district(df["district_no"])
    df["lease_no_norm"] = normalize_lease_no(df["lease_no"])

    df["lease_key"] = (
        df["oil_gas_code_norm"] + "_" +
        df["district_no_norm"] + "_" +
        df["lease_no_norm"]
    )

In [9]:
del df
prod_disp[["oil_gas_code", "district_no", "lease_no", "lease_key"]].head()

,oil_gas_code,district_no,lease_no,lease_key
0,O,08,09073,O_08_09073
1,O,10,27510,O_10_27510
2,O,10,27510,O_10_27510
3,O,10,27510,O_10_27510
4,O,10,27541,O_10_27541


In [10]:
prod_disp = prod_disp.drop(columns = ['oil_gas_code', 'district_no', 'lease_no','oil_gas_code_norm',
       'district_no_norm', 'lease_no_norm', ])


No we can combine it with the productionm data. Notice that if there is 5 wells in a lease this will create five rows for that lease for that month with the same production values. But that does not mean that each well in the lease produced the full lease volume. So the output of this part should be handled with care..

In [11]:
prod_well_points = prod_disp.merge(
    well_coord[
        [
            "lease_key",
            "api8",
            "well_no",
            "county_name",
            "longitude",
            "latitude",
            "geometry"
        ]
    ],
    on="lease_key",
    how="left"
)

prod_well_points = gpd.GeoDataFrame(
    prod_well_points,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Original production rows:", len(prod_disp))
print("Production rows after well join:", len(prod_well_points))
print(f"Production rows with well coordinates: {prod_well_points['geometry'].notna().mean():.2%}")


Original production rows: 7994195
Production rows after well join: 54727666
Production rows with well coordinates: 99.58%


In [12]:
del prod_disp
del well_coord

In [13]:
prod_well_points[['date','oil_sold_total_bbl','lease_key','longitude', 'latitude']].head(25)

,date,oil_sold_total_bbl,lease_key,longitude,latitude
0,2026-03-01,165,O_08_09073,-99.192087,33.016091
1,2026-03-01,165,O_08_09073,-99.191986,33.017324
2,2026-03-01,165,O_08_09073,-99.193200,33.017262
3,2026-03-01,165,O_08_09073,-99.194401,33.017304
4,2026-03-01,165,O_08_09073,-99.193118,33.018362
5,2026-03-01,165,O_08_09073,-99.195491,33.017285
6,2026-03-01,165,O_08_09073,-99.196405,33.017332
7,2026-03-01,165,O_08_09073,-99.197370,33.017478
8,2026-03-01,165,O_08_09073,-99.195963,33.016566
9,2026-03-01,165,O_08_09073,-99.200092,33.015225


We have a different file where we counted the number of wells in each lease. We can merge it with this one and the divide the dispensing values by number of wells.

In [14]:
well_num = pd.read_parquet("../../../../data/raw/texas/cleaned_data/wells_per_lease.parquet")
well_num.columns

Index(['lease_key', 'n_wells_with_coordinates', 'lease_latitude',
       'lease_longitude'],
      dtype='str')

In [15]:
well_num = well_num.drop(columns=['lease_latitude',
       'lease_longitude'])
well_num.columns

Index(['lease_key', 'n_wells_with_coordinates'], dtype='str')

In [16]:
prod_well_points = prod_well_points.merge(
    well_num,
    on="lease_key",
    how="left"
)


In [17]:
del well_num

In [18]:
prod_well_points.columns

Index(['field_no', 'operator_no', 'operator_name', 'oil_pipeline_bbl',
       'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl',
       'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl',
       'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl',
       'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf',
       'csgd_transmission_mcf', 'csgd_processing_plant_mcf',
       'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf',
       'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf',
       'csgd_no_disp_code_mcf', 'oil_sold_total_bbl',
       'total_vented_flared_mcf', 'date', 'lease_key', 'api8', 'well_no',
       'county_name', 'longitude', 'latitude', 'geometry',
       'n_wells_with_coordinates'],
      dtype='str')

In [19]:
prod_cols = ['oil_pipeline_bbl',
       'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl',
       'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl',
       'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl',
       'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf',
       'csgd_transmission_mcf', 'csgd_processing_plant_mcf',
       'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf',
       'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf',
       'csgd_no_disp_code_mcf', 'oil_sold_total_bbl',
       'total_vented_flared_mcf',]

In [20]:
prod_well_points[prod_cols] = prod_well_points[prod_cols].div(prod_well_points['n_wells_with_coordinates'],axis = 0)

In [21]:
prod_well_points[['date','oil_sold_total_bbl','lease_key','latitude','longitude','n_wells_with_coordinates']].head(5)

,date,oil_sold_total_bbl,lease_key,latitude,longitude,n_wells_with_coordinates
0,2026-03-01,9.705882,O_08_09073,33.016091,-99.192087,17.0
1,2026-03-01,9.705882,O_08_09073,33.017324,-99.191986,17.0
2,2026-03-01,9.705882,O_08_09073,33.017262,-99.193200,17.0
3,2026-03-01,9.705882,O_08_09073,33.017304,-99.194401,17.0
4,2026-03-01,9.705882,O_08_09073,33.018362,-99.193118,17.0


In [23]:
prod_well_points = prod_well_points.drop(columns = 'n_wells_with_coordinates')

In [24]:
prod_well_points.drop(columns="geometry").to_parquet(
    output_folder / "prod_per_well_approx.parquet",
    index=False
)

prod_well_points.to_parquet(
    output_folder / "prod_per_well_approx.geoparquet"
)